In [1]:
# 📥 Download March 2024 data
import requests
from tqdm import tqdm

file = 'green_tripdata_2024-03.parquet'
url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/{file}'
save_path = f'data/{file}'

resp = requests.get(url, stream=True)
with open(save_path, "wb") as f:
    for chunk in tqdm(resp.iter_content(), desc="Downloading", total=int(resp.headers["Content-Length"])):
        f.write(chunk)


Downloading: 100%|██████████████████████████████████████████████████████████████████████| 1372372/1372372 [00:05<00:00, 233179.87it/s]


In [2]:
# 🧼 Load and preview March 2024 data
import pandas as pd

mar_data = pd.read_parquet('data/green_tripdata_2024-03.parquet')
mar_data.shape  # 🔍 This gives the answer to Q1

(57457, 20)

In [3]:
# ✅ Setup column mapping as before
from evidently import ColumnMapping

num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

column_mapping = ColumnMapping(
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features,
    target=None
)

In [4]:
# ✅ Define updated report with additional metrics
from evidently.metrics import (
    ColumnDriftMetric,
    DatasetDriftMetric,
    DatasetMissingValuesMetric,
    ColumnQuantileMetric
)
from evidently.report import Report

report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric(),
    ColumnQuantileMetric(column_name='fare_amount', quantile=0.5)  # ✅ this is the new metric
])

In [8]:
import datetime

max_median_fare = 0.0

for day in range(1, 31):
    current_data = mar_data.loc[
        mar_data.lpep_pickup_datetime.between(
            datetime.datetime(2024, 3, day),
            datetime.datetime(2024, 3, day + 1),
            inclusive="left"
        )
    ].copy()

    current_data['lpep_pickup_datetime'] = pd.to_datetime(current_data['lpep_pickup_datetime'], errors='coerce')
    current_data = current_data.dropna(subset=['lpep_pickup_datetime'])

    daily_report = Report(metrics=[
        ColumnQuantileMetric(column_name='fare_amount', quantile=0.5)
    ], timestamp=datetime.datetime(2024, 3, day))

    daily_report.run(reference_data=None, current_data=current_data, column_mapping=column_mapping)
    result = daily_report.as_dict()

    # ✅ Correct access
    median = result['metrics'][0]['result']['current']['value']
    print(f"March {day:02d}: Median fare_amount = {median}")

    if median > max_median_fare:
        max_median_fare = median

print(f"\n✅ Maximum daily median fare_amount in March 2024 = {max_median_fare}")


March 01: Median fare_amount = 13.5
March 02: Median fare_amount = 13.5
March 03: Median fare_amount = 14.2
March 04: Median fare_amount = 12.8
March 05: Median fare_amount = 13.5
March 06: Median fare_amount = 12.8
March 07: Median fare_amount = 13.5
March 08: Median fare_amount = 13.5
March 09: Median fare_amount = 13.5
March 10: Median fare_amount = 14.2
March 11: Median fare_amount = 12.8
March 12: Median fare_amount = 13.5
March 13: Median fare_amount = 13.5
March 14: Median fare_amount = 14.2
March 15: Median fare_amount = 13.5
March 16: Median fare_amount = 14.2
March 17: Median fare_amount = 13.5
March 18: Median fare_amount = 13.5
March 19: Median fare_amount = 13.5
March 20: Median fare_amount = 12.8
March 21: Median fare_amount = 13.5
March 22: Median fare_amount = 13.5
March 23: Median fare_amount = 12.8
March 24: Median fare_amount = 14.2
March 25: Median fare_amount = 13.5
March 26: Median fare_amount = 13.5
March 27: Median fare_amount = 13.5
March 28: Median fare_amount

In [11]:
from evidently.metric_preset import DataQualityPreset
from evidently.ui.workspace import Workspace
from evidently.ui.dashboards import (
    DashboardPanelPlot,
    PanelValue,
    PlotType,
    ReportFilter,
)
from evidently.renderers.html_widgets import WidgetSize


In [14]:
from evidently.metric_preset import DataQualityPreset
from evidently.ui.workspace import Workspace
from evidently.ui.dashboards import DashboardPanelPlot, PanelValue, PlotType, ReportFilter
from evidently.renderers.html_widgets import WidgetSize

import datetime

# Open or create workspace
ws = Workspace("workspace")

# Create new project
project = ws.create_project("NYC Taxi Data Quality Project")
project.description = "Monitoring taxi duration prediction - expanded with fare_amount median"
project.save()


Project(id=UUID('019798a2-1a71-788e-a08e-3cc02057d414'), name='NYC Taxi Data Quality Project', description='Monitoring taxi duration prediction - expanded with fare_amount median', dashboard=DashboardConfig(name='NYC Taxi Data Quality Project', panels=[], tabs=[], tab_id_to_panel_ids={}), team_id=None, org_id=None, date_from=None, date_to=None, created_at=datetime.datetime(2025, 6, 22, 17, 14, 9, 905890), version='1')

In [15]:
from evidently.metrics import ColumnQuantileMetric
from evidently.report import Report

report = Report(
    metrics=[ColumnQuantileMetric(column_name='fare_amount', quantile=0.5)],
    timestamp=datetime.datetime(2024, 3, 1)
)
report.run(reference_data=None, current_data=mar_data, column_mapping=column_mapping)

ws.add_report(project.id, report)


In [16]:
project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Median Fare Amount (Q=0.5)",
        values=[
            PanelValue(
                metric_id="ColumnQuantileMetric",
                field_path="current.value",
                legend="fare_amount Q0.5"
            )
        ],
        plot_type=PlotType.LINE,
        size=WidgetSize.HALF
    )
)
project.save()

Project(id=UUID('019798a2-1a71-788e-a08e-3cc02057d414'), name='NYC Taxi Data Quality Project', description='Monitoring taxi duration prediction - expanded with fare_amount median', dashboard=DashboardConfig(name='NYC Taxi Data Quality Project', panels=[DashboardPanelPlot(type='evidently:dashboard_panel:DashboardPanelPlot', id=UUID('019798a2-5bc7-7890-809d-482976b2dff0'), title='Median Fare Amount (Q=0.5)', filter=ReportFilter(metadata_values={}, tag_values=[], include_test_suites=False), size=<WidgetSize.HALF: 1>, values=[PanelValue(field_path='current.value', metric_id='ColumnQuantileMetric', metric_fingerprint=None, metric_args={}, legend='fare_amount Q0.5')], plot_type=<PlotType.LINE: 'line'>)], tabs=[], tab_id_to_panel_ids={}), team_id=None, org_id=None, date_from=None, date_to=None, created_at=datetime.datetime(2025, 6, 22, 17, 14, 9, 905890), version='1')

In [18]:
import json
import os

# Ensure folder exists
os.makedirs("dashboards", exist_ok=True)

# Save dashboard config
with open("dashboards/grafana_dashboards.json", "w") as f_out:
    json.dump(project.dashboard.dict(), f_out, indent=2, default=str)

print("✅ Dashboard saved as 'dashboards/grafana_dashboards.json'")

✅ Dashboard saved as 'dashboards/grafana_dashboards.json'
